In [1]:
# this file will create AUC_graphs and AUC_data directories
# this code is the final version of the code that creates the area under the curve and data



In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG (SLP-MNIST)
# =========================
BASE_DIR_ROOT = r"C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
BATCH_DIR_TEMPLATE = "p-percentage_{:.1f}\\batch_size_{}"

BATCH_SIZES = [64, 1024, 60000]  # add/remove batch sizes as needed
PRUNING_PERCENTAGES = [round(x * 0.1, 1) for x in range(0, 11)]  # 0.0 → 1.0

# =========================
# OUTPUT DIRECTORIES
# =========================
AUC_GRAPH_DIR = os.path.join(BASE_DIR_ROOT, "AUC_graph")
AUC_DATA_DIR = os.path.join(BASE_DIR_ROOT, "AUC_data")
os.makedirs(AUC_GRAPH_DIR, exist_ok=True)
os.makedirs(AUC_DATA_DIR, exist_ok=True)
print(f"[INFO] Graphs → {AUC_GRAPH_DIR}")
print(f"[INFO] Data → {AUC_DATA_DIR}")

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12
})

# Maximum CE cap
CE_MAX = np.log(10)  # ≈ 2.3026

# =========================
# DEFINE COLORS
# =========================
COLOR_LIST = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#800080",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#B9D9EB", "#17becf"
]

# =========================
# MAIN LOOP
# =========================
all_auc_records = []  # single list for both CE types

for bs in BATCH_SIZES:

    avg_dfs = {}

    # -------------------------
    # LOAD AVERAGED FILES
    # -------------------------
    for p in PRUNING_PERCENTAGES:
        folder = os.path.join(BASE_DIR_ROOT, BATCH_DIR_TEMPLATE.format(p, bs))
        file_path = os.path.join(folder, f"averaged_runs_p_{p}_bs_{bs}.csv")

        if not os.path.isfile(file_path):
            print(f"[WARNING] File not found: {file_path}")
            continue

        df = pd.read_csv(file_path)
        # Cap CE values
        df["Avg_CE_Train"] = np.minimum(df["Avg_CE_Train"], CE_MAX)
        df["Avg_CE_Test"] = np.minimum(df["Avg_CE_Test"], CE_MAX)

        avg_dfs[p] = df

    if not avg_dfs:
        print(f"[WARNING] No data found for BS={bs}")
        continue

    # -------------------------
    # Find lowest max Batch_Number (exclude 100%)
    # -------------------------
    non100_dfs = {p: df for p, df in avg_dfs.items() if p != 1.0}
    if non100_dfs:
        lowest_max_batch = min(df["Batch_Number"].max() for df in non100_dfs.values())
    else:
        lowest_max_batch = min(df["Batch_Number"].max() for df in avg_dfs.values())

    # -------------------------
    # PLOT FUNCTION
    # -------------------------
    def plot_and_save_ce(avg_dfs, bs, ce_column, title):

        plt.figure(figsize=(10, 5))
        plt.title(f"{title} (BS={bs})")
        plt.xlabel("Batch Number")
        plt.ylabel(title)

        for i, p in enumerate(sorted(avg_dfs.keys(), reverse=True)):
            df = avg_dfs[p]
            color = COLOR_LIST[i % len(COLOR_LIST)]

            if p == 1.0:
                ce_value = df[ce_column].iloc[-1]
                ce_value = min(ce_value, CE_MAX)
                x = np.array([0, lowest_max_batch])
                y = np.array([ce_value, ce_value])
                auc = ce_value * lowest_max_batch

                plt.plot(x, y, color=color, linewidth=2.5,
                         label=f"P%={int(p*100)} | AUC={auc:.2f}")

            else:
                df_trunc = df[df["Batch_Number"] <= lowest_max_batch]
                x = df_trunc["Batch_Number"].to_numpy()
                y = np.minimum(df_trunc[ce_column].to_numpy(), CE_MAX)  # cap CE

                mask = np.isfinite(x) & np.isfinite(y)
                x, y = x[mask], y[mask]

                if len(x) == 0:
                    print(f"[WARNING] NaNs only for P%={p}, BS={bs}")
                    continue

                auc = np.trapezoid(y, x)

                plt.plot(x, y, color=color, linewidth=2.0,
                         label=f"P%={int(p*100)} | AUC={auc:.2f}")
                plt.fill_between(x, y, alpha=0.2, color=color)

            # Append to global records (single CSV)
            all_auc_records.append([bs, p, ce_column, auc])

        plt.grid(True)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
        plt.tight_layout()

        # SAVE FIGURE
        fig_path = os.path.join(AUC_GRAPH_DIR, f"{ce_column}_BS_{bs}.png")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"[SAVED] {fig_path}")

    # -------------------------
    # Plot Train & Test
    # -------------------------
    plot_and_save_ce(avg_dfs, bs, "Avg_CE_Train", "Average CE Train")
    plot_and_save_ce(avg_dfs, bs, "Avg_CE_Test", "Average CE Test")

# -------------------------
# SAVE SINGLE CSV FOR ALL
# -------------------------
if all_auc_records:
    auc_df = pd.DataFrame(all_auc_records, columns=["Batch_Size", "Pruning_Percentage", "CE_Type", "AUC"])
    csv_path = os.path.join(AUC_DATA_DIR, "AUC_ALL_BS.csv")
    auc_df.to_csv(csv_path, index=False)
    print(f"[SAVED] {csv_path}")

print("\n✅ All AUC graphs and combined data saved successfully.")


[INFO] Graphs → C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\AUC_graph
[INFO] Data → C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\AUC_data
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\AUC_graph\Avg_CE_Train_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\AUC_graph\Avg_CE_Test_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\AUC_graph\Avg_CE_Train_BS_1024.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\AUC_graph\Avg_CE_Test_BS_1024.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\AUC_graph\Avg_CE_Train_BS_60000.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\AUC_graph\Avg_CE_Test_BS_60000.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\S